In [ ]:
import numpy as np
import librosa
import soundfile as sf
import os


FRAME_MS = 25
HOP_MS   = 15
DIRECTORY = "C:/Users/admin/Documents/PythonFile/Project/Voice-Activity-Detect/data/raw/musan/speech/librivox"
DIRECTORY_SAVE = "C:/Users/admin/Documents/PythonFile/Project/Voice-Activity-Detect/data/processed/speech"

def short_time_energy(frames):
    return np.sum(frames ** 2, axis=0)

def frames_to_audio(frames, hop_len):
    frame_len, num_frames = frames.shape

    signal_len = frame_len + (num_frames - 1) * hop_len

    signal = np.zeros(signal_len)
    window_sum = np.zeros(signal_len)

    window = np.hanning(frame_len)

    for i in range(num_frames):
        start = i * hop_len
        end = start + frame_len

        signal[start:end] += frames[:, i] * window
        window_sum[start:end] += window

    nonzero = window_sum > 1e-8
    signal[nonzero] /= window_sum[nonzero]

    return signal

count = 1
for file in os.listdir(DIRECTORY):
    if file.lower().endswith(".wav"):
        path = os.path.join(DIRECTORY, file)
        print(path)
        y, sr = librosa.load(path, sr=None)
        print(count, sr)
        count += 1
        frame_len = int(FRAME_MS * sr / 1000)
        hop_len   = int(HOP_MS   * sr / 1000)
        frames = librosa.util.frame(
            y,
            frame_length=frame_len,
            hop_length=hop_len
        )
        energy = short_time_energy(frames)
        TH_E = np.mean(energy) * 0.1

        speech_mask = energy > TH_E
        speech_frames = frames[:, speech_mask]

        print("Total frames:", frames.shape[1])
        print("Speech frames:", speech_frames.shape[1])

        speech_audio = frames_to_audio(speech_frames, hop_len)

        sf.write(f"{DIRECTORY_SAVE}\{count - 1}.wav", speech_audio, sr)